# 03 · Explore millions of taxi records

Query a real month of NYC trips while returning small, inspectable summaries.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [NYC TLC trip records and taxi-zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This lesson uses the January 2024 yellow-taxi Parquet file. See the
[NYC Open Data terms of use](https://opendata.cityofnewyork.us/overview/#termsofuse)
and TLC's data dictionary and accuracy notice; do not treat public availability
as a warranty or apply the notebook's code license to the data.
The original Parquet download is roughly 50 MB and contains roughly three million
rows. The helper checks the pinned file digest. Queries use explicit result
limits; a small result does not imply a small source scan.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


## Inspect the source without loading all rows

Parquet metadata supplies an independent row count. Reading metadata here does
not execute an analytical query outside Kelvo.


In [ ]:
import pyarrow.parquet as pq
taxi = lab.taxi()
config = lab.catalog(sources=[{"id": "trips", "type": "parquet", "path": str(taxi)}])
parquet = pq.ParquetFile(taxi)
print({"source_bytes": taxi.stat().st_size,
       "rows": parquet.metadata.num_rows,
       "row_groups": parquet.metadata.num_row_groups})
print(parquet.schema_arrow)


In [ ]:
profile = lab.query(config, """
    SELECT COUNT(*)::BIGINT AS total,
           COUNT(*) FILTER (
               WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
                 AND tpep_pickup_datetime < TIMESTAMP '2024-02-01'
           )::BIGINT AS january_pickups,
           MIN(tpep_pickup_datetime) AS first_pickup,
           MAX(tpep_pickup_datetime) AS last_pickup,
           COUNT(*) FILTER (WHERE passenger_count IS NULL)::BIGINT AS missing_passengers
    FROM trips
""", sources=["trips"], max_rows=10)
row = profile.table().to_pylist()[0]
assert row["total"] == parquet.metadata.num_rows
assert 0 < row["january_pickups"] <= row["total"]
assert 0 <= row["missing_passengers"] <= row["total"]
display(profile.table().to_pandas())


## Inspect a bounded excerpt

`LIMIT` bounds returned rows. It does not by itself bound scan work, sort memory,
or native allocations. The helper also configures query memory and deadlines.


In [ ]:
excerpt = lab.query(config, """
    SELECT tpep_pickup_datetime, "PULocationID", "DOLocationID",
           trip_distance, total_amount
    FROM trips
    WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
      AND tpep_pickup_datetime < TIMESTAMP '2024-01-02'
    ORDER BY tpep_pickup_datetime, "PULocationID", "DOLocationID", total_amount, trip_distance
    LIMIT 20
""", sources=["trips"], max_rows=20)
assert excerpt.table().num_rows == 20
assert excerpt.stats["rows"] == 20
display(excerpt.table().to_pandas())
print({"elapsed_seconds": excerpt.elapsed_seconds, "query_stats": excerpt.stats})


The source contains outliers and occasional timestamps outside the nominal
month. Filter by the analytical period explicitly; do not assume every row in a
monthly file belongs to that period. These timings describe this run only and
are not a comparative benchmark.


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
